## Random Reaction Networks

#### Early implementation of random networks of reactions that are **thermodynamically possible**,
#### and **biologically plausible**

### TAGS :   "uniform compartment"

In [1]:
LAST_REVISED = "Oct. 20, 2026"
LIFE123_VERSION = "1.0.0rc9"         # Library version this experiment is based on

In [2]:
#import set_path            # Using MyBinder?  Uncomment this before running the next cell!
                            # Importing this module will add the project's home directory to sys.path

In [3]:
#import sys, os
#os.getcwd()
#sys.path.append("C:/some_path/my_env_or_install")   # CHANGE to the folder containing your venv or libraries installation!
# NOTE: If any of the imports below can't find a module, uncomment the lines above, or try:  import set_path

import ipynbname
from IPython.display import IFrame

from life123 import check_version, RandomReactionNetwork, ReactionSimulator

In [4]:
check_version(LIFE123_VERSION)    # To check compatibility

OK


In [5]:
# Initialize the HTML logging (for the graphics)
log_file = ipynbname.name() + ".log.htm"    # Use the notebook base filename for the log file
                                            # IN CASE OF PROBLEMS, set manually to any desired name
log_file

'random_networks_1.log.htm'

## Set up Random Network

In [6]:
net = RandomReactionNetwork(n_species=4, n_rxns=7, seed=47242315, thermodynamic_ruggedness=0.2)

In [7]:
net.species_data.get_all_species_ids()

['A', 'B', 'C', 'D']

#### The above species id's were auto-generated

In [8]:
rxns = net.get_reaction_data()          # Object of type "ReactionRegistry"

In [9]:
rxns.number_of_reactions()

7

In [10]:
rxns.describe_reactions(detail="minimal")

Number of reactions: 7
0: C <-> A + D
1: 2 C <-> D
2: B + C <-> D
3: B <-> 2 D
4: D <-> A + B
5: A + D <-> B
6: D + C <-> A
^-----  Species involved in the above reactions: {"A", "B", "C", "D"}


In [11]:
rxns.describe_reactions(detail="summary")

Number of reactions: 7
0: C <-> A + D
        Unimolecular decomposition reaction, with Reaction Model: "mass action"
        Thermodynamics:  (delta_H = -16.262 kJ/mol | delta_S = 52.421 J/(mol·K) | delta_G = -31.891 kJ/mol | K_eq = 3.8646e+05 | Temp = 25 C)
        Kinetics: 1 DERIVED reaction (as used by simulator)
                  [1]   C <-> A + D
                        Type: "mass action"   (kF = 3,783.6 | kR = 0.0097904 | K = 3.8646e+05 | reversible = True)
1: 2 C <-> D
        Bimolecular synthesis reaction, with Reaction Model: "mass action"   (Reaction ID 1)
        Thermodynamics:  (delta_H = -8.9575 kJ/mol | delta_S = -59.996 J/(mol·K) | delta_G = 8.9301 kJ/mol | K_eq = 0.027259 | Temp = 25 C)
        Kinetics: 1 DERIVED reaction (as used by simulator)
                  [1]   2 C <-> D
                        Type: "mass action"   (kF = 0.36677 | kR = 13.455 | K = 0.027259 | reversible = True)
2: B + C <-> D
        Bimolecular synthesis reaction, with Reaction Model: "ma

## NOTE ON UNITS: _delta_H_ and _delta_G_ are in **kJ/mol**  ;  _delta_S_ are in **J/(mol·K)**

#### Investigate the random values that were assigned to the reactions (in a manner to be consistent with thermodynamics, for example across groups of reactions):

In [12]:
H_list = [rxn.get_thermodynamics().delta_H
          for rxn in net.get_all_random_reactions()]    # rxn is an object of type "SimulationReaction"
H_list   # kJ/mol

[-16.262093710585276,
 -8.95754366062873,
 -9.227408711383152,
 -8.674548248998072,
 -4.417410753915934,
 16.5319587613397,
 -11.027545220012897]

In [13]:
S_list = [rxn.get_thermodynamics().delta_S
          for rxn in net.get_all_random_reactions()]    # rxn is an object of type "SimulationReaction"
S_list   # J/(mol·K)

[52.42053331751184,
 -59.995529512556274,
 -81.98455786740513,
 10.735492461211042,
 64.14701422039786,
 -38.78571157151433,
 -37.290078736939144]

In [14]:
G_list = [rxn.get_thermodynamics().delta_G
          for rxn in net.get_all_random_reactions()]    # rxn is an object of type "SimulationReaction"
G_list   # kJ/mol

[-31.89127571920143,
 8.930123463539921,
 15.216287216783687,
 -11.875335326308143,
 -23.542843043727558,
 28.095918666386694,
 0.09049175540550891]

In [15]:
# Verify G_list
[H_list[i] - (25+273.15) * S_list[i] / 1000 for i in range(len(H_list))]

[-31.89127571920143,
 8.930123463539921,
 15.216287216783687,
 -11.875335326308143,
 -23.542843043727558,
 28.095918666386694,
 0.09049175540550891]

In [16]:
# Instantiate the simulator with our species and reactions
sim = ReactionSimulator(reaction_registry=rxns, method="heun")   # analytic

In [17]:
# Send a plot of the network of reactions to the HTML log file
sim.plot_reaction_network(log_file=log_file)

[GRAPHIC ELEMENT SENT TO FILE `random_networks_1.log.htm`]


In [18]:
IFrame(log_file, width=1200, height=700)         # You may also open the above log file in a browser

# Run a simulation

In [19]:
sim.set_system_state([31, 4, 18, 82])

In [20]:
sim.get_history()

,SYSTEM TIME,A,B,C,D,step,caption
0,0.0,31,4,18,82,,Set system state


In [21]:
sim.system.get_conc_as_dict()

{'A': 31, 'B': 4, 'C': 18, 'D': 82}

In [22]:
sim.system.get_system_conc()

array([31,  4, 18, 82])

In [25]:
sim.single_compartment_react(initial_step=0.00001, target_end_time=1.)   # Using defaults for all other parameters

UFuncTypeError: Cannot cast ufunc 'add' output from dtype('float64') to dtype('int32') with casting rule 'same_kind'

In [27]:
sim.system.system_state

array([31,  4, 18, 82])

In [26]:
sim.system_time

0.0

In [ ]:
sim.plot_history(show_intervals=True)